In [ ]:
import xarray as xr
import scipy
import xskillscore as xs
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs

import concurrent.futures
from concurrent.futures import ProcessPoolExecutor

import albedo_functions as af

import logging

In [ ]:
# Configurazione logging
log_file = "process_log.txt"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_file),  # Write to file
    ]
)

In [ ]:
plt.rcParams.update({
    'font.size': 20,          # Font di base
    'axes.titlesize': 20,     # Titolo del grafico
    'axes.labelsize': 20,     # Etichette degli assi
    'xtick.labelsize': 20,    # Etichette sull'asse x
    'ytick.labelsize': 20,    # Etichette sull'asse y
    'legend.fontsize': 20,    # Font della legenda
})

In [ ]:
exp_ctrl = 'a1ua'
exp_sens = 'a52o'
BASE_PATH = '/confess/dicarlo/02-confess/05-confess-post-data/'
SAVE_PATH = '/confess/dicarlo/cartella_ordinata/'

In [ ]:
def plots(season):
    logging.info(f'Starting {season}')
    try:
        lai_ctrl = xr.open_dataset(f'{SAVE_PATH}{exp_ctrl}_lai_trend_{season}_1999.nc')['slope']
        p_ctrl = xr.open_dataset(f'{SAVE_PATH}a1ua_lai_trend_p_{season}_1999.nc')
        lai_sens = xr.open_dataset(f'{SAVE_PATH}{exp_sens}_lai_trend_{season}_1999.nc')['slope']
        p_sens = xr.open_dataset(f'{SAVE_PATH}a52o_lai_trend_p_{season}_1999.nc')
        p_value_lai = xr.open_dataset(f'{SAVE_PATH}delta_lai_trend_p_{season}_1999.nc')

        lai_ctrl = af.land_seas_mask(lai_ctrl)
        lai_sens = af.land_seas_mask(lai_sens)

        levels=[-0.5, -0.4, -0.3, -0.2, -0.1, 0, 0.1, 0.2, 0.3, 0.4, 0.5]
        title = f'a) DCPP-CTRL'
        af.map_plot(af.fix_longitude(lai_ctrl*10), af.fix_longitude(p_ctrl.p), levels=levels, title=title, cmap='BrBG', antartica=False)
        plt.savefig(f'/confess/dicarlo/figure-decadale/a1ua_lai_slope_{season}_1999.png', dpi=600, bbox_inches = 'tight')
        logging.info(f'fig1')

        title = f'b) DCPP-SENS'
        af.map_plot(af.fix_longitude(lai_sens*10), af.fix_longitude(p_sens.p), levels=levels, title=title, cmap='BrBG', antartica=False)
        plt.savefig(f'/confess/dicarlo/figure-decadale/a52o_lai_slope_{season}_1999.png', dpi=600, bbox_inches = 'tight')
        logging.info(f'fig1')

        levels=[-0.6,-0.5,-0.4,-0.3,-0.2,-0.1,0,0.1,0.2,0.3,0.4,0.5,0.6]
        title=f'c) DCPP-SENS minus DCPP-CTRL'
        af.map_plot(af.fix_longitude(lai_sens*10 - lai_ctrl*10), af.fix_longitude(p_value_lai.p_delta), levels=levels,title=title, cmap='BrBG', antartica=False)
        plt.savefig(f'/confess/dicarlo/figure-decadale/delta_lai_slope_{season}_1999.png', dpi=600, bbox_inches = 'tight')
        logging.info(f'fig1')

        import matplotlib.image as mpimg
        img1 = mpimg.imread(f'/confess/dicarlo/figure-decadale/a1ua_lai_slope_{season}_1999.png')
        img2 = mpimg.imread(f'/confess/dicarlo/figure-decadale/a52o_lai_slope_{season}_1999.png')
        img3 = mpimg.imread(f'/confess/dicarlo/figure-decadale/delta_lai_slope_{season}_1999.png')
        af.triple_figure(img1, img2, img3, f'/confess/dicarlo/figure-decadale/fig2_lai_trend_{season}_1999.png')
        logging.info(f'fig2')
    except Exception as e:
        logging.error(f"Failed for {season}: {e}", exc_info=True)

In [ ]:
%%time
seasons=['DJF', 'JJA', 'MAM', 'SON']
futures=[]
with concurrent.futures.ProcessPoolExecutor() as executor:
    futures = [executor.submit(plots, i) for i in seasons]

    # Wait for all tasks to complete
    concurrent.futures.wait(futures)